# READ-WRITE FILE

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark =    SparkSession.builder \
    .appName("day12") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint",           "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",         "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key",         "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access",  "true") \
    .config("spark.hadoop.fs.s3a.impl",               "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.catalogImplementation",        "hive") \
    .config("hive.metastore.uris",                    "thrift://metastore:9083") \
    .config("spark.eventLog.enabled",                 "true") \
    .config("spark.eventLog.dir",                     "s3a://spark-event/") \
    .config("spark.sql.shuffle.partitions",           str(4)) \
    .config("spark.sql.adaptive.enabled",             "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

In [3]:
spark.sparkContext.setLogLevel("WARN")

transformation

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [5]:
# ── HIVE CHECK ────────────────────────────────────────────────────────────────
# local_db is auto-created by init-hive-db container on first startup
print("\n── Hive databases (local_db should appear) ────────────────────────────")
spark.sql("SHOW DATABASES").show()

spark.sql("SHOW TABLES IN local_db").show()


── Hive databases (local_db should appear) ────────────────────────────
+---------+
|namespace|
+---------+
|  default|
| local_db|
+---------+

+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
| local_db|       date_summary|      false|
| local_db|  sample_hive_table|      false|
| local_db|total_amount_ranked|      false|
+---------+-------------------+-----------+



In [6]:
try:
    yellow_taxi = spark.sql("SELECT * FROM local_db.sample_hive_table")
    print("[INFO] Loaded from Hive: local_db.sample_hive_table")
except Exception:
    yellow_taxi = spark.read.parquet("s3a://spark-warehouse/raw_data/save_from_notebook")
    print("[WARN] Hive table not found — loaded from Parquet. Run Day 9 first.")
 
taxi_zone = spark.read \
    .option("header", "true").option("inferSchema", "true") \
    .csv("s3a://metadata-rawdata/taxi_zone_lookup.csv")
 
 
print(f"taxi : {yellow_taxi.count()} rows")
print(f"taxi_zone : {taxi_zone.count()} rows")

[INFO] Loaded from Hive: local_db.sample_hive_table
taxi : 300000 rows
taxi_zone : 265 rows


In [10]:
# ── Write modes ───────────────────────────────────────────────────────────────
# overwrite — replaces everything at the path
# append    — adds new files alongside existing ones
# ignore    — silently skips if path already exists
# error     — raises exception if path already exists (default)

#yellow_taxi.write.mode("overwrite").parquet("s3a://raw-data/output/taxi_overwrite/")

In [ ]:
# ── Compression options ───────────────────────────────────────────────────────
# snappy  — fast read/write, moderate compression (default, best for most cases)
# gzip    — slower write, better compression (use for cold/archival data)
# zstd    — best balance of speed + compression (Spark 3.x+, use this in production)
#yellow_taxi.write.mode("overwrite").option("compression", "zstd").parquet("s3a://raw-data/output/taxi_overwrite_zstd/")

In [7]:
# ── Read ─────────────────────────────────────────────────────────────────────
#df = spark.read.parquet("s3a://raw-data/output/taxi_overwrite/")

# Read multiple files with wildcard — Spark unions them automatically
taxi1 = spark.read.parquet("s3a://raw-data/yellow_tripdata_2025-*.parquet")

# If files have slightly different schemas across months, merge them
taxi2 = spark.read \
    .option("mergeSchema", "true") \
    .parquet("s3a://raw-data/yellow_tripdata_2025-*.parquet")

taxi1.printSchema()
taxi2.printSchema()

print(f"taxi1: {taxi1.count()} taxi2: {taxi2.count()}")

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: ti

partition helps spark to speed up to look for certain data     
can check result from day9 lesson

In [8]:
# When you filter on the partition column, Spark reads ONLY that folder
# This is called partition pruning — visible in .explain() as PartitionFilters
spark.read \
    .parquet("s3a://spark-warehouse/raw_data/save_from_notebook") \
    .filter(F.col("tpep_pickup_date") == "2025-05-02") \
    .explain()
# Look for: PartitionFilters: [isnotnull(tpep_pickup_date#xx), (tpep_pickup_date#xx = 2025-05-02)]
# This means Spark never opened other date folders

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [VendorID#276,tpep_pickup_datetime#277,tpep_dropoff_datetime#278,passenger_count#279L,trip_distance#280,RatecodeID#281L,store_and_fwd_flag#282,PULocationID#283,DOLocationID#284,payment_type#285L,fare_amount#286,extra#287,mta_tax#288,tip_amount#289,improvement_surcharge#290,total_amount#291,congestion_surcharge#292,Airport_fee#293,cbd_congestion_fee#294,total_amount_thb#295,is_valid_passenger_count#296,tpep_pickup_date#297] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[s3a://spark-warehouse/raw_data/save_from_notebook], PartitionFilters: [isnotnull(tpep_pickup_date#297), (tpep_pickup_date#297 = 2025-05-02)], PushedFilters: [], ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passen...




In [9]:
# Multi-column partitioning — for time-series data
# Partition by year then month — the standard pattern for event/log data
taxi1.withColumn("year",  F.year("tpep_pickup_datetime")) \
    .withColumn("month", F.month("tpep_pickup_datetime")) \
    .write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("s3a://spark-warehouse/raw_data/multiple_col_partition/")

In [10]:
# Query one month — Spark skips all other year/month folders
spark.read.parquet("s3a://spark-warehouse/raw_data/multiple_col_partition/") \
     .filter((F.col("year") == 2025) & (F.col("month") == 10)) \
     .count()

4428708

Choosing partition columns   
- Low cardinality category, not High cardinality because a number of partitions wii be too many
- Date hierarchy: year + month not date
- not Boolean / status as there is too few data -> unbalanced
- not Free text -> unpredictable cardinality

**many partitions reduce speed to search data among partitions**

In [11]:
# Check partition count before writing
print(f"Partitions before: {taxi1.rdd.getNumPartitions()}")

Partitions before: 10


In [12]:
# coalesce(n) — reduces partitions WITHOUT a full shuffle
# Merges local partitions on each executor — cheap but can be unbalanced
# Only use to REDUCE partition count, not increase it
taxi1.limit(500000).coalesce(1).write.mode("overwrite").parquet("s3a://spark-warehouse/raw_data/test_partition/taxi_coalesce/")
# ↑ Safe for 200 rows. Dangerous for 50M rows (all data pulled to one executor)


In [13]:
# repartition(n) — full shuffle across network, perfectly balanced
# Use when INCREASING partitions, or when data is skewed
taxi1.limit(500000).repartition(8).write.mode("overwrite").parquet("s3a://spark-warehouse/raw_data/test_partition/taxi_8files/")

In [14]:
# repartition by column — forces one partition per distinct value
# Useful before partitionBy to avoid 1-file-per-partition fragmentation
taxi1.limit(500000).repartition(F.col("VendorID")).write \
    .mode("overwrite") \
    .partitionBy("VendorID") \
    .parquet("s3a://spark-warehouse/raw_data/test_partition/vendorID/")

In [15]:
spark.stop()

# ICEBERG
need to re-read to use docker exec to `spark-master` on `README.md`

In [41]:
spark_config = {
    "spark.jars.packages":                      "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
    "spark.sql.extensions":                     "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    "spark.sql.catalog.spark_catalog":          "org.apache.iceberg.spark.SparkSessionCatalog",
    "spark.hadoop.fs.s3a.endpoint":             "http://minio:9000",
    "spark.hadoop.fs.s3a.access.key":           "minioadmin",
    "spark.hadoop.fs.s3a.secret.key":           "minioadmin",
    "spark.hadoop.fs.s3a.path.style.access":    "true",
    "spark.hadoop.fs.s3a.impl":                 "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.sql.catalog.spark_catalog.type":     "hive",
    "hive.metastore.uris":                      "thrift://metastore:9083",
    "spark.eventLog.enabled":                   "true",
    "spark.eventLog.dir":                       "s3a://spark-event/",
    "spark.sql.shuffle.partitions":             "20"
}


In [42]:
spark2 = SparkSession.builder \
    .appName("day12_Iceberg") \
    .master("spark://spark-master:7077") \
    .config(map=spark_config) \
    .enableHiveSupport() \
    .getOrCreate()

In [43]:
spark2.sparkContext.setLogLevel("WARN")

transformation

In [ ]:
taxi1 = spark2.read.parquet("s3a://raw-data/yellow_tripdata_2025-*.parquet")

In [8]:
# writeTo API — Iceberg's native write interface
# create() fails if table already exists
taxi1.limit(600000).writeTo("local_db.ytaxi_iceberg") \
         .using("iceberg") \
         .create()

Py4JJavaError: An error occurred while calling o76.create.
: org.apache.iceberg.exceptions.NotFoundException: Failed to open input stream for file: s3a://spark-warehouse/hive/local_db/taxi_iceberg/metadata/00001-864f9146-2a04-4ba1-b1cb-a30d50bcf82b.metadata.json
	at org.apache.iceberg.hadoop.HadoopInputFile.newStream(HadoopInputFile.java:185)
	at org.apache.iceberg.TableMetadataParser.read(TableMetadataParser.java:279)
	at org.apache.iceberg.TableMetadataParser.read(TableMetadataParser.java:273)
	at org.apache.iceberg.BaseMetastoreTableOperations.lambda$refreshFromMetadataLocation$0(BaseMetastoreTableOperations.java:182)
	at org.apache.iceberg.BaseMetastoreTableOperations.lambda$refreshFromMetadataLocation$1(BaseMetastoreTableOperations.java:201)
	at org.apache.iceberg.util.Tasks$Builder.runTaskWithRetry(Tasks.java:413)
	at org.apache.iceberg.util.Tasks$Builder.runSingleThreaded(Tasks.java:219)
	at org.apache.iceberg.util.Tasks$Builder.run(Tasks.java:203)
	at org.apache.iceberg.util.Ta

In [9]:
# createOrReplace() for idempotent writes
taxi1.limit(600000).writeTo("local_db.ytaxi_iceberg") \
         .using("iceberg") \
         .createOrReplace()

In [ ]:
# Append — Iceberg creates a new snapshot, keeps old one intact
#new_rows.writeTo("local_db.xxx_iceberg").append()

# Dynamic overwrite — replaces only partitions present in the incoming data
# Only partitions that have the values to modify
#new_data.writeTo("local_db.xxx_iceberg").overwritePartitions()

In [14]:
# Read — same as any Hive table
df = spark2.read.table("local_db.ytaxi_iceberg")
print(f"Rows: {df.count()}")

Rows: 600000


In [17]:
# SQL works natively
spark2.sql("""
    SELECT COUNT(VendorID) AS cnt
    FROM local_db.ytaxi_iceberg
""").show()

+------+
|   cnt|
+------+
|600000|
+------+



In [18]:
print(f"Partitions: {df.rdd.getNumPartitions()}")

Partitions: 1


can't run group by on spark.sql because of error 134 or OOM issue    


In [16]:
# Inspect metadata via special table suffixes
spark2.sql("SELECT * FROM local_db.ytaxi_iceberg.snapshots").show()
spark2.sql("SELECT * FROM local_db.ytaxi_iceberg.history").show()
spark2.sql("SELECT * FROM local_db.ytaxi_iceberg.files").show(truncate=False)

+--------------------+-------------------+---------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+---------+---------+--------------------+--------------------+
|2026-04-15 08:18:...|6090645697689919618|     NULL|   append|s3a://spark-wareh...|{spark.app.id -> ...|
|2026-04-15 08:19:...|9017763553542882950|     NULL|   append|s3a://spark-wareh...|{spark.app.id -> ...|
|2026-04-15 08:42:...|2483669701530814978|     NULL|   append|s3a://spark-wareh...|{spark.app.id -> ...|
+--------------------+-------------------+---------+---------+--------------------+--------------------+

+--------------------+-------------------+---------+-------------------+
|     made_current_at|        snapshot_id|parent_id|is_current_ancestor|
+--------------------+-------------------+---------+-------------------+
|2026-04-15 08:18:...|6090645697689919618|   

In [ ]:
# MERGE or UPSERT data into existing iceberg
#spark.sql("""
#    MERGE INTO local_db.xxx_iceberg AS target
#    USING xxx_temp_table AS source
#    ON target.xx_id = source.xx_id
#    WHEN MATCHED THEN
#        UPDATE SET target.value1 = source.value1
#    WHEN NOT MATCHED THEN
#        INSERT (xx_id, name, value1, foreign_id)
#        VALUES (source.xx_id, source.name, source.value1, source.foreign_id)
#""")

In [19]:
# Every write creates a new snapshot — query any past version
spark2.sql("SELECT * FROM local_db.ytaxi_iceberg.snapshots").show()

+--------------------+-------------------+---------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+---------+---------+--------------------+--------------------+
|2026-04-15 08:18:...|6090645697689919618|     NULL|   append|s3a://spark-wareh...|{spark.app.id -> ...|
|2026-04-15 08:19:...|9017763553542882950|     NULL|   append|s3a://spark-wareh...|{spark.app.id -> ...|
|2026-04-15 08:42:...|2483669701530814978|     NULL|   append|s3a://spark-wareh...|{spark.app.id -> ...|
+--------------------+-------------------+---------+---------+--------------------+--------------------+



In [21]:
# Query by snapshot ID (get the ID from .snapshots output)
spark2.read \
    .option("snapshot-id", "6090645697689919618") \
    .table("local_db.ytaxi_iceberg") \
    .count()



600000

In [24]:
# Query by timestamp
#spark2.read.option("as-of-timestamp", "2026-04-15 08:20:00").table("local_db.ytaxi_iceberg").count() # doesn't work because of NumberFormatException 

# choose the closest snapshot to show
spark2.sql("""
    SELECT count(VendorID) 
    FROM local_db.ytaxi_iceberg FOR TIMESTAMP AS OF '2026-04-15 08:20:00'
""").show()

+---------------+
|count(VendorID)|
+---------------+
|         600000|
+---------------+



In [28]:
spark2.sql("SELECT * FROM local_db.ytaxi_iceberg.history").show()

+--------------------+-------------------+---------+-------------------+
|     made_current_at|        snapshot_id|parent_id|is_current_ancestor|
+--------------------+-------------------+---------+-------------------+
|2026-04-15 08:18:...|6090645697689919618|     NULL|              false|
|2026-04-15 08:19:...|9017763553542882950|     NULL|              false|
|2026-04-15 08:42:...|2483669701530814978|     NULL|               true|
+--------------------+-------------------+---------+-------------------+



In [25]:
# SQL time travel syntax (get schema)
spark2.sql("""
    SELECT * FROM local_db.ytaxi_iceberg
    FOR SYSTEM_VERSION AS OF 9017763553542882950
""")

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double]

In [29]:
# Roll back to a previous snapshot # choose is_current_ancestor = True
# because writing new data to replace = reset history. use Append to able to get is_current_ancestor = True
spark2.sql("""
    CALL spark_catalog.system.rollback_to_snapshot(
        'local_db.ytaxi_iceberg',
        2483669701530814978
    )
""")

DataFrame[previous_snapshot_id: bigint, current_snapshot_id: bigint]

In [30]:
spark2.stop()

# Delta Lake

In [40]:
from delta import configure_spark_with_delta_pip

In [4]:
spark_config2 = {
    "spark.hadoop.fs.s3a.endpoint":             "http://minio:9000",
    "spark.hadoop.fs.s3a.access.key":           "minioadmin",
    "spark.hadoop.fs.s3a.secret.key":           "minioadmin",
    "spark.hadoop.fs.s3a.path.style.access":    "true",
    "spark.hadoop.fs.s3a.impl":                 "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.sql.shuffle.partitions":             "4",
    "spark.eventLog.enabled":                   "true",
    "spark.eventLog.dir":                       "s3a://spark-event/",
    "spark.sql.extensions":                     "io.delta.sql.DeltaSparkSessionExtension",
    "spark.sql.catalog.spark_catalog":          "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    "spark.sql.catalogImplementation":          "hive",
    "hive.metastore.uris":                      "thrift://metastore:9083",
    "spark.sql.adaptive.enabled":               "true"
    
}

spark3 = configure_spark_with_delta_pip( \
    SparkSession.builder \
    .appName("day12_delta") \
    .master("spark://spark-master:7077") \
    .config(map=spark_config2) \
    .enableHiveSupport() \
).getOrCreate()

In [5]:
spark3.sparkContext.setLogLevel("WARN")

In [6]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F



In [7]:
spark3.sql("SHOW TABLES IN local_db").show()

+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
| local_db|       date_summary|      false|
| local_db|  sample_hive_table|      false|
| local_db|total_amount_ranked|      false|
| local_db|      ytaxi_iceberg|      false|
+---------+-------------------+-----------+



In [10]:
spark3.sql("DROP TABLE  local_db.taxi_iceberg")

DataFrame[]

In [12]:
taxi = spark3.sql("SELECT * FROM local_db.sample_hive_table")

delta_path = "s3a://spark-warehouse/raw_data/delta/example/"

# Write Delta
taxi.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)


In [13]:
# Read Delta
df = spark3.read.format("delta").load(delta_path)
print(f"Rows: {df.count()}")



Rows: 300000


In [14]:
# See the transaction log
DeltaTable.forPath(spark3, delta_path).history().show()

+-------+-------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|userId|userName|operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      0|2026-04-15 12:59:31|  NULL|    NULL|    WRITE|{mode -> Overwrit...|NULL|    NULL|     NULL|       NULL|  Serializable|        false|{numFiles -> 2, n...|        NULL|Apache-Spark/3.5....|
+-------+-------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+



In [15]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+---------------------+------------+--------------------+-----------+------------------+----------------+------------------------+----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|total_amount_thb|is_valid_passenger_count|tpep_pickup_date|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+---------------------+------------+--------------------+-----------+------------------+----------------+------------------------+----------

In [22]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- total_amount_thb: double (nullable = true)
 |-- is_valid_passenger_count: boolean (nullable = true)
 |-- tpep_pickup_date: 

In [23]:
from datetime import datetime, date

data = [
    (
        2, 
        datetime.strptime("2025-01-01 00:18:38", "%Y-%m-%d %H:%M:%S"), # tpep_pickup_datetime
        datetime.strptime("2025-01-01 00:35:13", "%Y-%m-%d %H:%M:%S"), # tpep_dropoff_datetime
        3, 2.0, 1, "N", 229, 244, 1, 50.0, 1.0, 0.5, 0.0, 1.0, 55.0, 0.0, 0.0, 0.0, 2000.0, 
        True,               # is_valid_passenger_count (Boolean)
        date(2025, 1, 1)    # tpep_pickup_date (Date)
    )
]

In [24]:

updates = spark3.createDataFrame(data, df.schema)
len(updates.columns)

22

In [25]:
delta_table = DeltaTable.forPath(spark3, delta_path)



In [26]:
delta_table.alias("target").merge(
    updates.alias("source"),
    "target.VendorID = source.VendorID AND target.tpep_pickup_datetime = source.tpep_pickup_datetime AND target.tpep_dropoff_datetime = source.tpep_dropoff_datetime" 
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

In [27]:
# See all versions
DeltaTable.forPath(spark3, delta_path).history() \
    .select("version", "timestamp", "operation").show()



+-------+-------------------+---------+
|version|          timestamp|operation|
+-------+-------------------+---------+
|      1|2026-04-15 14:09:14|    MERGE|
|      0|2026-04-15 12:59:31|    WRITE|
+-------+-------------------+---------+



In [28]:
# Read a specific version
v0 = spark3.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(delta_path)
print(f"Version 0: {v0.count()} rows")



Version 0: 300000 rows


In [34]:
# Read by timestamp
print("Version 1: ", spark3.read.format("delta") \
    .option("timestampAsOf", "2026-04-15 14:09:14") \
    .load(delta_path).count(), " rows")



Version 1:  300000  rows


In [35]:
# Restore to previous version
DeltaTable.forPath(spark3, delta_path).restoreToVersion(0)

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]

In [ ]:
spark3.stop()

# exercise

**exercise1**

In [39]:
exer1 = spark.read.parquet("s3a://spark-warehouse/raw_data/multiple_col_partition/")

exer1.filter(
    (F.col("year") == 2025) & (F.col("month") == 10)
).explain()

exer1.count()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [VendorID#6369,tpep_pickup_datetime#6370,tpep_dropoff_datetime#6371,passenger_count#6372L,trip_distance#6373,RatecodeID#6374L,store_and_fwd_flag#6375,PULocationID#6376,DOLocationID#6377,payment_type#6378L,fare_amount#6379,extra#6380,mta_tax#6381,tip_amount#6382,tolls_amount#6383,improvement_surcharge#6384,total_amount#6385,congestion_surcharge#6386,Airport_fee#6387,cbd_congestion_fee#6388,year#6389,month#6390] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[s3a://spark-warehouse/raw_data/multiple_col_partition], PartitionFilters: [isnotnull(year#6389), isnotnull(month#6390), (year#6389 = 2025), (month#6390 = 10)], PushedFilters: [], ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passen...




48722602

Exercise2    

too many partition folders to query will create overhead in metadata file size and reduce speed to query

**exercise3**     
UPSERT WITH DELETE

In [ ]:
updates_with_deletes = spark2.createDataFrame([
    (1,  "Somchai", 95000.0, "ENG", "active"),
    (5,  "Siriwan",  0.0,    "HR",  "deleted"),
], ["emp_id", "name", "salary", "dept_id", "status"])

updates_with_deletes.createOrReplaceTempView("updates")

spark2.sql("""
    MERGE INTO local_db.employees_iceberg AS target
    USING updates AS source
    ON target.emp_id = source.emp_id
    WHEN MATCHED AND source.status = 'deleted' THEN DELETE
    WHEN MATCHED AND source.status = 'active'  THEN UPDATE SET *
    WHEN NOT MATCHED AND source.status = 'active' THEN INSERT *
""")

spark2.sql("SELECT * FROM local_db.employees_iceberg.snapshots").show()

**exercise4**

In [44]:
# iceberg
spark2.read.option("multiLine", "true").json("s3a://spark-warehouse/hive/local_db/ytaxi_iceberg/metadata/00000-55157da9-6125-4cc7-bb26-94a678ae6808.metadata.json").show()

+-----------------+-------------------+---------------------+---------------+--------------+--------------+-----------------+--------------------+---------------+--------------------+------------+---------------+--------------------+------------+--------------------+--------------------+--------------------+--------------------+-----------+----------+--------------------+
|current-schema-id|current-snapshot-id|default-sort-order-id|default-spec-id|format-version|last-column-id|last-partition-id|last-sequence-number|last-updated-ms|            location|metadata-log|partition-specs|partition-statistics|  properties|                refs|             schemas|        snapshot-log|           snapshots|sort-orders|statistics|          table-uuid|
+-----------------+-------------------+---------------------+---------------+--------------+--------------+-----------------+--------------------+---------------+--------------------+------------+---------------+--------------------+------------+----

In [45]:
# delta lake
spark3.read.json("s3a://spark-warehouse/raw_data/delta/ytaxi/_delta_log/00000000000000000001.json").show()

+--------------------+--------------------+
|                 add|          commitInfo|
+--------------------+--------------------+
|                NULL|{Apache-Spark/3.5...|
|{true, 1776262154...|                NULL|
+--------------------+--------------------+



In [ ]:
spark.stop()


In [46]:
spark2.stop()

In [47]:
spark3.stop()